# WR GRU Baseline Model - Predict Receiving Yards Next Game

This notebook builds a simple baseline GRU (Gated Recurrent Unit) model to predict an NFL wide receiver's receiving yards in their next game, using the last 5 games as input sequences.

In [1]:
# ── Imports ─────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow version:", tf.__version__)
print("NumPy version:", np.__version__)


TensorFlow version: 2.21.0
NumPy version: 2.4.0


## 1. Load Dataset and Inspect Columns

In [ ]:
# Load the WR game-level dataset
DATA_PATH = "../../../data/processed/wr_all_weeks.csv"
df_raw = pd.read_csv(DATA_PATH)

print("Shape:", df_raw.shape)
print("\nColumns:")
print(df_raw.columns.tolist())
print("\nDtypes:")
print(df_raw.dtypes)
print("\nFirst 3 rows:")
df_raw.head(3)


FileNotFoundError: [Errno 2] No such file or directory: '../data/processed/wr_all_weeks.csv'

## 2. Identify Key Columns

From the inspection we can confirm:
- Player identifier: `receiver_player_id`
- Season: `season` column (integer year)
- Week: extracted from `game_id` (format `YYYY_WW_TEAM_TEAM`)
- Target: `receiving_yards`

In [3]:
# Column roles — derived from inspection above
PLAYER_COL  = "receiver_player_id"   # unique player identifier
SEASON_COL  = "season"               # integer season year
TARGET_COL  = "receiving_yards"      # prediction target

# Extract week number from game_id string (format: "YYYY_WW_HOME_AWAY")
df_raw["week"] = df_raw["game_id"].str.split("_").str[1].astype(int)

print("Unique seasons:", sorted(df_raw[SEASON_COL].unique()))
print("Week range:    ", df_raw["week"].min(), "–", df_raw["week"].max())
print("Target column  sample stats:")
print(df_raw[TARGET_COL].describe())


Unique seasons: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Week range:     1 – 22
Target column  sample stats:
count    46115.000000
mean        29.642047
std         31.758577
min        -17.000000
25%          7.000000
50%         19.000000
75%         43.000000
max        300.000000
Name: receiving_yards, dtype: float64


## 3. Sort Data by Player and Time Order

In [4]:
# Sort chronologically within each player so sequences are time-ordered
df = df_raw.sort_values(
    by=[PLAYER_COL, SEASON_COL, "week"],
    ascending=True
).reset_index(drop=True)

print(f"Rows after sort: {len(df)}")
print(df[[PLAYER_COL, SEASON_COL, "week", TARGET_COL]].head(10))


Rows after sort: 46115
  receiver_player_id  season  week  receiving_yards
0         00-0019596    2015    13             36.0
1         00-0019596    2017    21              0.0
2         00-0019596    2018    10              6.0
3         00-0019596    2022    10              0.0
4         00-0020337    2015     1             13.0
5         00-0020337    2015     2            150.0
6         00-0020337    2015     3            186.0
7         00-0020337    2015     4             24.0
8         00-0020337    2015     6            137.0
9         00-0020337    2015     7             78.0


## 4. Handle Missing Values

In [5]:
# Keep only numeric columns (drop object/string columns that can't be used as features)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# Exclude the derived 'week' column from features (it's only used for ordering)
# and keep TARGET_COL in the numeric set so it's accessible during windowing
feature_cols = [c for c in numeric_cols if c not in ("week",)]

print(f"Total numeric columns available as features: {len(feature_cols)}")

# Check missing values
missing = df[feature_cols].isnull().sum()
print("\nColumns with missing values:")
print(missing[missing > 0])

# Forward-fill within each player group (carry the last known value forward)
df[feature_cols] = df.groupby(PLAYER_COL)[feature_cols].transform(
    lambda grp: grp.ffill()
)

# Drop any rows that still have NaNs (e.g. first row of a player with no prior data)
df.dropna(subset=feature_cols, inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"\nRows after NaN handling: {len(df)}")
print("Remaining NaNs in features:", df[feature_cols].isnull().sum().sum())


Total numeric columns available as features: 95

Columns with missing values:
Series([], dtype: int64)

Rows after NaN handling: 46115
Remaining NaNs in features: 0


## 5. Create Sliding Window Sequences

For every player, we slide a window of 5 consecutive games across their history.  
- X[i] = feature matrix of games t, t+1, …, t+4 (shape: `5 × n_features`)  
- y[i] = `receiving_yards` in game t+5 (the next game after the window)

In [6]:
TIMESTEPS = 5   # number of past games used as input

X_list, y_list = [], []

for player_id, player_df in df.groupby(PLAYER_COL):
    # All feature values for this player in chronological order
    values = player_df[feature_cols].values   # shape: (n_games, n_features)
    target = player_df[TARGET_COL].values     # shape: (n_games,)

    # Only create sequences if the player has enough games
    if len(values) < TIMESTEPS + 1:
        continue

    for i in range(len(values) - TIMESTEPS):
        X_list.append(values[i : i + TIMESTEPS])      # window of 5 games
        y_list.append(target[i + TIMESTEPS])           # next game's yards

X_all = np.array(X_list, dtype=np.float32)   # (samples, 5, n_features)
y_all = np.array(y_list, dtype=np.float32)   # (samples,)

print(f"Total sequences created : {X_all.shape[0]}")
print(f"X shape (samples, timesteps, features): {X_all.shape}")
print(f"y shape                               : {y_all.shape}")


Total sequences created : 39125
X shape (samples, timesteps, features): (39125, 5, 95)
y shape                               : (39125,)


## 6. Train/Test Split and Scale Features

We split before fitting the scaler to prevent data leakage from the test set into the training distribution.

In [7]:
# 80/20 split — shuffle=False preserves temporal ordering within each sequence
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.20, random_state=42, shuffle=True
)

n_features = X_train.shape[2]
print(f"Train samples : {len(X_train)}")
print(f"Test  samples : {len(X_test)}")
print(f"Features      : {n_features}")

# ── Scale features ─────────────────────────────────────────────────────────
# Flatten to 2-D, fit on train only, then reshape back to 3-D
scaler = StandardScaler()

X_train_2d = X_train.reshape(-1, n_features)
X_test_2d  = X_test.reshape(-1, n_features)

X_train_scaled = scaler.fit_transform(X_train_2d).reshape(-1, TIMESTEPS, n_features)
X_test_scaled  = scaler.transform(X_test_2d).reshape(-1, TIMESTEPS, n_features)

print("\nScaled X_train shape:", X_train_scaled.shape)
print("Scaled X_test  shape:", X_test_scaled.shape)


Train samples : 31300
Test  samples : 7825
Features      : 95

Scaled X_train shape: (31300, 5, 95)
Scaled X_test  shape: (7825, 5, 95)


## 7. Build Tensors and Confirm Shapes

In [8]:
# Convert to TensorFlow tensors (Keras also accepts raw NumPy arrays)
X_train_t = tf.constant(X_train_scaled, dtype=tf.float32)
X_test_t  = tf.constant(X_test_scaled,  dtype=tf.float32)
y_train_t = tf.constant(y_train,        dtype=tf.float32)
y_test_t  = tf.constant(y_test,         dtype=tf.float32)

print("Tensor shapes")
print(f"  X_train : {X_train_t.shape}   ← (samples, timesteps=5, features={n_features})")
print(f"  X_test  : {X_test_t.shape}")
print(f"  y_train : {y_train_t.shape}")
print(f"  y_test  : {y_test_t.shape}")


Tensor shapes
  X_train : (31300, 5, 95)   ← (samples, timesteps=5, features=95)
  X_test  : (7825, 5, 95)
  y_train : (31300,)
  y_test  : (7825,)


## 8. Build the GRU Model

Architecture:
- Input: `(5, n_features)`
- GRU: 32 units
- Dense: 1 neuron (linear output = predicted yards)

In [9]:
tf.random.set_seed(42)

model = keras.Sequential([
    keras.Input(shape=(TIMESTEPS, n_features)),
    layers.GRU(32),                    # single GRU layer with 32 units
    layers.Dense(1),                   # linear output — predicted receiving yards
], name="WR_GRU_Baseline")

model.summary()


Model: "WR_GRU_Baseline"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, 32)             │        12,384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,417 (48.50 KB)

 Trainable params: 12,417 (48.50 KB)

 Non-trainable params: 0 (0.00 B)

## 9. Compile and Train the Model

In [10]:
model.compile(
    optimizer=keras.optimizers.Adam(),
    loss="mae",          # Mean Absolute Error as the training loss
    metrics=["mae"],
)

history = model.fit(
    X_train_t, y_train_t,
    batch_size=32,
    epochs=20,
    validation_split=0.10,   # 10 % of training data used for validation
    verbose=1,
)


Epoch 1/20
881/881 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 23.3889 - mae: 23.3889 - val_loss: 21.0262 - val_mae: 21.0262
Epoch 2/20
881/881 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 20.3055 - mae: 20.3055 - val_loss: 20.0448 - val_mae: 20.0448
Epoch 3/20
881/881 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 19.5001 - mae: 19.5001 - val_loss: 19.8288 - val_mae: 19.8288
Epoch 4/20
881/881 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 19.0974 - mae: 19.0974 - val_loss: 19.7496 - val_mae: 19.7496
Epoch 5/20
881/881 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 18.8331 - mae: 18.8331 - val_loss: 19.7063 - val_mae: 19.7063
Epoch 6/20
881/881 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 18.6238 - mae: 18.6238 - val_loss: 19.7041 - val_mae: 19.7041
Epoch 7/20
881/881 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 18.4413 - mae: 18.4413 - val_loss: 19.7233 - val_mae: 19.7233
Epoch 8/20
881/881 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 18.2643 - mae: 18.2643 - val_loss: 19.7439 - val_mae: 19.7439
Epoch 9/20
881/881 ━━━━━

## 10. Evaluate the Model on the Test Set

In [11]:
# Generate predictions on the held-out test set
y_pred = model.predict(X_test_t, batch_size=32).flatten()

# ── Metrics ──────────────────────────────────────────────────────────────
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print("=" * 40)
print("  Test-set evaluation results")
print("=" * 40)
print(f"  MAE  : {mae:.4f}  yards")
print(f"  RMSE : {rmse:.4f}  yards")
print(f"  R²   : {r2:.4f}")
print("=" * 40)


245/245 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
  Test-set evaluation results
  MAE  : 20.3054  yards
  RMSE : 28.5405  yards
  R²   : 0.2590


## 11. Print Final Tensor Shapes and Example Predictions

In [12]:
# ── Final tensor shapes ───────────────────────────────────────────────────
print("Final tensor shapes:")
print(f"  X_train : {X_train_t.shape}")
print(f"  X_test  : {X_test_t.shape}")
print(f"  y_train : {y_train_t.shape}")
print(f"  y_test  : {y_test_t.shape}")

# ── Example predictions (first 10 test samples) ───────────────────────────
print("\nFirst 10 predictions vs actuals:")
comparison = pd.DataFrame({
    "Actual  (yards)": y_test[:10].round(1),
    "Predicted (yards)": y_pred[:10].round(1),
    "Error (yards)": (y_pred[:10] - y_test[:10]).round(1),
})
print(comparison.to_string(index=True))


Final tensor shapes:
  X_train : (31300, 5, 95)
  X_test  : (7825, 5, 95)
  y_train : (31300,)
  y_test  : (7825,)

First 10 predictions vs actuals:
   Actual  (yards)  Predicted (yards)  Error (yards)
0             57.0          60.900002       3.900000
1             17.0          11.600000      -5.400000
2             16.0          62.099998      46.099998
3             13.0          21.600000       8.600000
4              7.0          23.799999      16.799999
5              6.0          10.300000       4.300000
6             92.0          56.900002     -35.099998
7              7.0          11.300000       4.300000
8             40.0          35.299999      -4.700000
9             45.0          65.699997      20.700001
